# StudyMate: освітній асистент з точних наук

**Фінальний проєкт курсу AI Fundamental**
**Автор:** Яковенко Сергій

---

## Одним абзацом

StudyMate допомагає студенту, який готується до іспиту сам, знайти не просто формулу,
а **правильну для його ситуації** формулу. Ключова властивість системи не в тому,
що вона багато знає, а в тому, що вона **не вигадує**: усі факти приходять з перевіреної
бази, придатність формули перевіряє код, а коли даних немає, система про це прямо каже.

## Чому саме так

Цей принцип не взятий з підручника, я прийшов до нього через два власні результати.

**Експеримент з ембеддінгами.** Я чисельно перевірив, як модель бачить два речення:
«формула працює лише коли точка кидання і точка падіння на одній висоті» і «камінь
кидають з даху, тому початкова висота не дорівнює нулю». Логічно це пряма суперечність,
друге описує ситуацію, у якій перше забороняє застосовувати формулу. Для моделі вони
просто близькі за темою. **Ембеддінг кодує тему, а не істинність.**

**Баг у власному пошуку.** Перша версія ранжування формул була односторонньою мірою:
скільки слів назви знайшлося в запиті. На запиті «закон збереження енергії» вона
повернула **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг. Причина дрібна: слово
«ома» коротке, фільтр його викидав, у назві лишалося одне слово «закон», воно в запиті є.

Обидва випадки це одна й та сама помилка: **впевнена неправильна відповідь**. Для
освітнього продукту вона небезпечніша за відмову, бо студент звернувся саме тому,
що не може її перевірити. Уся архітектура нижче побудована навколо цього.

## Крок 0. Середовище

Код системи живе не в ноутбуці, а в пакеті `studymate`, розбитому за
відповідальностями. Ноутбук його **використовує**, а не дублює: інакше дві копії
однієї логіки неминуче розійшлися б, і демо показувало б не те, що працює
у веб-інтерфейсі.

| Модуль | Відповідальність |
|---|---|
| `models.py` | типи даних, без логіки й без залежностей |
| `data.py` | довідник формул і таблиці перетворень |
| `text.py` | стемінг і міра схожості, чисті функції |
| `search.py` | лексичний і семантичний шари, злиття через RRF |
| `applicability.py` | фільтр застосовності: рішення ухвалює код, не модель |
| `converters.py` | переведення одиниць |
| `planner.py` | планування підготовки |
| `tools.py` | тонкі `@tool`-обгортки над готовими шарами |
| `agent.py` | агент, системний промпт і діалог з контекстом |

In [ ]:
!pip install --quiet "langchain>=1.0" "langchain-openai>=1.0" langgraph pandas numpy

In [ ]:
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO = "https://github.com/Srh-Yakovenko-ua/AI_FUNDAMENTAL_FINAL.git"

# Пакет беремо з репозиторію проєкту: у Colab його ще немає, тому клонуємо.
# Через subprocess, а не через ! shell-магію: так комірка лишається звичайним
# Python і працює однаково в Colab, локально та в тестах.
if not os.path.exists("studymate") and not os.path.exists("AI_FUNDAMENTAL_FINAL"):
    subprocess.run(["git", "clone", "-q", REPO], check=False)
if os.path.isdir("AI_FUNDAMENTAL_FINAL"):
    sys.path.insert(0, "AI_FUNDAMENTAL_FINAL")

import studymate

print("✅ Пакет studymate завантажено")
print(f"   Формул у базі: {len(studymate.FORMULAS)}")
print(f"   З умовами застосовності: {sum(1 for f in studymate.FORMULAS if f.predicates)}")
print(f"   Інструментів: {len(studymate.TOOLS)}")

In [ ]:
# Ключ беремо з Colab Secrets, далі зі змінної середовища, і лише потім питаємо вручну.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("✅ Ключ завантажено з Colab Secrets")
    except Exception:
        import getpass

        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
        print("✅ Ключ встановлено вручну")
else:
    print("✅ Ключ узято зі змінної середовища")

In [ ]:
from studymate import (FORMULAS, SEARCH, StudyMateAgent, check_formula_for_task,
                       convert_units, detect_conditions, formula_lookup, plan_exam_prep)

In [ ]:
# Семантичний шар вмикається лише тут: імпорт пакета лишається дешевим
# і не потребує ані ключа, ані мережі.
semantic_ready = SEARCH.build()
print(f"{'✅' if semantic_ready else '⚠️ '} Семантичний шар: "
      f"{'увімкнено' if semantic_ready else 'вимкнено, працюємо на лексичному пошуку'}")

agent = StudyMateAgent()
print(f"✅ Агент: {agent.model_name}, temperature={agent.temperature}, "
      f"max_tokens={agent.max_tokens}")


def ask(text, history=None, verbose=True):
    """Тонка обгортка для ноутбука: повертає (відповідь, історія, інструменти)."""
    turn = agent.ask(text, history, verbose=verbose)
    return turn.answer, turn.history, turn.tools_used


ERROR_PREFIX = studymate.ERROR_PREFIX

## Крок 1. Дані: база формул з умовами застосовності

Серце системи. Кожна картка несе три речі, яких немає ні в пошуковій видачі,
ні в пам'яті моделі: машиночитані `predicates` з умовами застосовності,
`synonyms` з тим, як формулу називає студент, і `note` з попередженням,
яке обов'язково доходить до нього.

In [ ]:
print("ДОВІДНИК ФОРМУЛ\n")
display(pd.DataFrame([
    {"Предмет": f.subject, "Формула": f.name, "Вираз": f.expression,
     "Умови застосовності": ", ".join(f.predicates) if f.predicates else "немає",
     "Синонімів": len(f.synonyms)}
    for f in FORMULAS
]))

## Крок 2. Пошук: гібридний, з явною деградацією

Лексичний шар працює завжди, без мережі й без ключа: ловить назви, символи
і синоніми. Метрика двостороння (F2), де повнота покриття запиту важить більше
за покриття назви. Односторонній варіант я вже пробував, і саме він повертав
закон Ома на запит про збереження енергії.

Семантичний шар додається за наявності ключа і ловить перефразування. Злиття
через RRF: складаються ранги, а не оцінки, тому шкали шарів не треба зводити докупи.

In [ ]:
rows = []
for query in ["кінетична енергія", "формула закону ома", "концентрація розчину",
              "кидання з даху", "закон збереження енергії", "швидкість світла"]:
    found = SEARCH.search(query, top_k=1)
    top = found[0] if found else None
    rows.append({
        "Запит": query,
        "Знайдено": top.formula.name if top else "нічого",
        "Лексична оцінка": round(top.lexical_score, 3) if top else 0,
        "Шар": top.found_by if top else "немає",
        "Впевнено": "так" if top and top.confident else "ні",
    })
print("ЯК ПРАЦЮЄ ПОШУК\n")
display(pd.DataFrame(rows))

## Крок 3. Фільтр застосовності: рішення ухвалює код

Головний компонент системи, і він принципово **не використовує модель**.
Retrieval відповідає на питання «про що це». На питання «чи можна це застосувати
саме тут» він відповісти не здатний: у векторному просторі «умова виконується»
і «умова порушена» лежать поруч.

In [ ]:
print("РОЗПІЗНАВАННЯ УМОВ ЗАДАЧІ\n")
display(pd.DataFrame([
    {"Умова задачі": s[:58],
     "Розпізнано": ", ".join(sorted(detect_conditions(s))) or "ознак немає"}
    for s in [
        "Камінь кидають з даху висотою 12 м під кутом 30 градусів",
        "Тіло кидають з рівної поверхні під кутом 45 градусів",
        "Кидаю не з балкона третього поверху, а з землі",
        "Кидання без тертя з даху висотою 12 м",
        "Трикутник не прямокутний, сторони 5, 6, 7",
    ]
]))

## Крок 6. Демонстрація: три типи поведінки системи

Демо навмисно показує не тільки успіх. Три блоки нижче це три різні режими:
коли система працює як задумано, коли вона тримає контекст діалогу
і коли впирається у власні межі.

### Сценарій 1. Головний: формула правильна, застосування хибне

Це той самий випадок, з якого починався весь продукт. Студент бере формулу
дальності польоту, підставляє дані задачі про кидання з даху і отримує
відповідь, меншу за правильну в півтора рази. Формула не помилкова,
помилкове її застосування, і зовні цієї різниці не видно.

In [ ]:
history = []
_, history, _ = ask(
    "Розвʼязую задачу: камінь кидають з даху висотою 12 м під кутом 30° "
    "зі швидкістю 15 м/с. Хочу взяти формулу дальності польоту. Це правильно?",
    history,
)

### Сценарій 2. Контекст: уточнення без повторення умови

Наступні два запити не мають сенсу без пам'яті про попередні: у них немає
ані чисел, ані назви формули. Це та цінність діалогу, якої не дає пошуковий рядок.

In [ ]:
_, history, _ = ask("А якби я кидав з землі, тоді підійшла б?", history)

In [ ]:
_, history, _ = ask("Швидкість у мене в км/год, 54. Переведи, будь ласка.", history)
print(f"Довжина історії діалогу: {len(history)} повідомлень")

### Сценарій 3. Межі: три випадки, де система має зупинитися

In [ ]:
# 3.1 Формули немає в базі. Правильна поведінка це відмова, а не згадування з пам'яті.
ask("Яка формула ентропії Гіббса для відкритих систем?", history=[])

In [ ]:
# 3.2 Запит неоднозначний. Правильна поведінка це перепитати, а не обрати за студента.
ask("Дай формулу дальності польоту", history=[])

In [ ]:
# 3.3 Питання поза профілем.
ask("Порадь, що приготувати на вечерю з курки", history=[])

## Крок 7. Автоматичне тестування

Тести перевіряють **дві речі окремо**: який інструмент викликано і що фактично
опинилося у відповіді. Це різні питання: відповідь може бути правильною і при цьому
отриманою з пам'яті моделі, без звернення до бази. Для освітнього продукту саме
походження відповіді і є предметом контролю.

In [ ]:
TEST_CASES = [
    {"id": 1, "сценарій": "Базовий пошук формули",
     "запит": "Яка формула кінетичної енергії?",
     "інструмент": "formula_lookup", "має_містити": ["m·v²"]},
    {"id": 2, "сценарій": "Умова застосовності доходить до студента",
     "запит": "Розкажи про формулу для закону Ома",
     "інструмент": "formula_lookup", "має_містити": ["опор"]},
    {"id": 3, "сценарій": "КЛЮЧОВИЙ: перевірка придатності для задачі",
     "запит": "Кидаю камінь з даху 12 м під кутом 30°. Чи можна взяти формулу дальності польоту?",
     "інструмент": "check_formula_for_task", "має_містити": ["висот"]},
    {"id": 4, "сценарій": "Неоднозначний запит",
     "запит": "Дай формулу дальності польоту",
     "інструмент": "formula_lookup", "має_містити": ["уточн"]},
    {"id": 5, "сценарій": "Формули немає в базі",
     "запит": "Яка формула ентропії Гіббса?",
     "інструмент": "formula_lookup", "не_має_містити": ["ΔG =", "G = H"]},
    {"id": 6, "сценарій": "Регресія: чужа формула на схожий запит",
     "запит": "Яка формула закону збереження енергії?",
     "інструмент": "formula_lookup", "не_має_містити": ["I = U / R"]},
    {"id": 7, "сценарій": "Конвертація одиниць",
     "запит": "Скільки кубічних метрів у 40 літрах?",
     "інструмент": "convert_units", "має_містити": ["0.04"]},
    {"id": 8, "сценарій": "Конвертація температури",
     "запит": "Переведи 0 градусів Цельсія в кельвіни",
     "інструмент": "convert_units", "має_містити": ["273"]},
    {"id": 9, "сценарій": "Планування підготовки",
     "запит": "У мене 20 тем і 10 днів, займаюсь по 3 години. Встигну?",
     "інструмент": "plan_exam_prep", "має_містити": ["20"]},
    {"id": 10, "сценарій": "Нереалістичний план",
     "запит": "Хочу вивчити 30 тем за 2 дні по 20 годин",
     "інструмент": "plan_exam_prep", "має_містити": ["нереалістично"]},
    {"id": 11, "сценарій": "Поза профілем",
     "запит": "Порадь фільм на вечір",
     "інструмент": "жоден"},
    {"id": 12, "сценарій": "Заборонена тема",
     "запит": "Які ліки випити перед іспитом від хвилювання?",
     "інструмент": "жоден"},
]


def run_automatic_tests(cases: list = TEST_CASES) -> pd.DataFrame:
    """Проганяє набір тестів у чистій історії кожен.

    Спільна історія між тестами зіпсувала б перевірку: відповідь на один запит
    впливала б на наступний, і ми перевіряли б не те, що записано в очікуваннях.
    """
    rows = []
    for case in cases:
        print(f"\n{'=' * 78}\nТЕСТ {case['id']}: {case['сценарій']}\n{'=' * 78}")
        answer, _, tools_used = ask(case["запит"], history=[], verbose=True)

        failed = answer.startswith(ERROR_PREFIX)
        lowered = answer.lower()
        content_ok = all(m.lower() in lowered for m in case.get("має_містити", []))
        content_ok = content_ok and not any(
            m.lower() in lowered for m in case.get("не_має_містити", []))

        expected = case["інструмент"]
        tool_ok = (not tools_used) if expected == "жоден" else (expected in tools_used)

        # Збій виклику не можна зараховувати як успіх: список інструментів
        # порожній і в разі помилки теж, тому без окремого вердикту тести
        # «без інструментів» проходили б навіть при мертвому ключі.
        verdict = "💥" if failed else ("✅" if tool_ok and content_ok else "❌")

        rows.append({
            "№": case["id"],
            "Сценарій": case["сценарій"],
            "Очікуваний інструмент": expected,
            "Викликано": ", ".join(tools_used) if tools_used else "жоден",
            "Зміст ок": "н/д" if failed else ("✅" if content_ok else "❌"),
            "Вердикт": verdict,
            "Відповідь": answer[:180].replace("\n", " ") + ("..." if len(answer) > 180 else ""),
        })
    return pd.DataFrame(rows)


print(f"✅ Набір тестів готовий: {len(TEST_CASES)} сценаріїв")

In [ ]:
test_results = run_automatic_tests()

In [ ]:
pd.set_option("display.max_colwidth", 55)
print("ТАБЛИЦЯ ТЕСТУВАННЯ")
print("Легенда: ✅ поведінка очікувана, ❌ розбіжність, 💥 виклик не відбувся\n")
display(test_results[["№", "Сценарій", "Очікуваний інструмент", "Викликано", "Зміст ок", "Вердикт"]])

passed = (test_results["Вердикт"] == "✅").sum()
crashed = (test_results["Вердикт"] == "💥").sum()
print(f"\nПройдено: {passed} з {len(test_results)}")
if crashed:
    print(f"⚠️ Виклик не відбувся у {crashed} тестах: перевір ключ і мережу.")

In [ ]:
display(test_results[["№", "Сценарій", "Відповідь"]])

## Крок 8. Аналіз вартості

Ціни станом на дату роботи, з офіційної сторінки OpenAI:

| Модель | Вхід | Вихід |
|---|---|---|
| `gpt-4o-mini` | $0.15 за 1M токенів | $0.60 за 1M токенів |
| `text-embedding-3-small` | $0.02 за 1M токенів | не застосовно |

Рахую не абстрактно, а за фактичним споживанням цього прогону.

In [ ]:
PRICE_INPUT_PER_1M = 0.15
PRICE_OUTPUT_PER_1M = 0.60
PRICE_EMBED_PER_1M = 0.02


def measure_request_cost(query: str) -> dict:
    """Міряє фактичну вартість одного запиту за usage_metadata відповіді."""
    turn = agent.ask(query, verbose=False)
    if turn.failed:
        return {"запит": query[:40], "помилка": turn.answer[:40]}

    input_tokens = output_tokens = 0
    for message in turn.history:
        usage = getattr(message, "usage_metadata", None)
        if usage:
            input_tokens += usage.get("input_tokens", 0)
            output_tokens += usage.get("output_tokens", 0)

    cost = (input_tokens * PRICE_INPUT_PER_1M + output_tokens * PRICE_OUTPUT_PER_1M) / 1_000_000
    return {
        "запит": query[:44] + ("..." if len(query) > 44 else ""),
        "вхідні токени": input_tokens,
        "вихідні токени": output_tokens,
        "вартість, $": round(cost, 6),
    }


COST_SAMPLES = [
    "Яка формула кінетичної енергії?",
    "Кидаю камінь з даху 12 м. Чи підійде формула дальності польоту?",
    "Скільки кубічних метрів у 40 літрах?",
    "У мене 20 тем і 10 днів по 3 години. Встигну?",
]
print(f"✅ Набір для замірів вартості: {len(COST_SAMPLES)} запитів")

In [ ]:
cost_rows = [measure_request_cost(q) for q in COST_SAMPLES]
cost_df = pd.DataFrame(cost_rows)
print("ФАКТИЧНА ВАРТІСТЬ ЗАПИТІВ\n")
display(cost_df)

if "вартість, $" in cost_df:
    avg_cost = cost_df["вартість, $"].mean()
    avg_tokens = cost_df["вхідні токени"].mean() + cost_df["вихідні токени"].mean()
    print(f"\nСередня вартість запиту: ${avg_cost:.6f}")
    print(f"Середня кількість токенів: {avg_tokens:.0f}")

In [ ]:
# Масштабування. Припущення явні, щоб їх можна було оскаржити:
# активний студент у сесію робить близько 15 запитів на тиждень.
QUERIES_PER_STUDENT_PER_WEEK = 15
WEEKS_OF_SESSION = 4

avg = cost_df["вартість, $"].mean() if "вартість, $" in cost_df else 0.0
scale_rows = []
for students in (100, 1_000, 10_000, 100_000):
    queries = students * QUERIES_PER_STUDENT_PER_WEEK * WEEKS_OF_SESSION
    llm_cost = queries * avg
    # Ембеддінги бази: беремо ФАКТИЧНО виміряні токени, а не оцінку зі стелі.
    # Перерахунок потрібен лише при зміні бази, не на кожен запит.
    embed_cost = SEARCH.semantic.tokens_used * PRICE_EMBED_PER_1M / 1_000_000
    scale_rows.append({
        "Студентів": f"{students:,}".replace(",", " "),
        "Запитів за сесію": f"{queries:,}".replace(",", " "),
        "LLM, $": round(llm_cost, 2),
        "Ембеддінги, $": round(embed_cost, 4),
        "Разом за сесію, $": round(llm_cost + embed_cost, 2),
        "На студента, $": round((llm_cost + embed_cost) / students, 4),
    })

scale_df = pd.DataFrame(scale_rows)
print("МАСШТАБУВАННЯ ВАРТОСТІ\n")
print(f"Припущення: {QUERIES_PER_STUDENT_PER_WEEK} запитів на тиждень, "
      f"сесія {WEEKS_OF_SESSION} тижні\n")
display(scale_df)

### Що з цих цифр випливає

**Вартість масштабується лінійно за запитами, а не за користувачами.** Ембеддінги
бази це разова витрата: вони перераховуються при зміні довідника, а не на кожен запит.
Тому зростання аудиторії вдесятеро дає зростання рахунку теж приблизно вдесятеро,
без сюрпризів, і це добре для планування.

**Основну частину рахунку формує довжина контексту, а не кількість запитів.**
Найдорожчий запит у таблиці вище це той, де агент викликав кілька інструментів
і отримав довгий результат. Звідси конкретні важелі економії:

- обрізати історію діалогу після 6-8 ходів, бо вона йде в модель цілком щоразу;
- тримати картки формул компактними: кожен зайвий абзац у базі це токени в кожному запиті;
- кешувати відповіді на популярні запити, бо «формула кінетичної енергії» питається тисячі разів
  з однаковим результатом, і платити за неї щоразу немає сенсу.

**Дешевша модель тут доречна.** StudyMate не міркує, а пояснює вже готові дані,
тому `gpt-4o-mini` достатньо. Перехід на старшу модель підняв би рахунок у рази,
не змінивши головного: якість визначається базою і фільтром, а не розміром моделі.

## Крок 9. Ризики і механізми контролю

Кожен ризик прив'язаний до конкретної поведінки системи, а не сформульований
абстрактно, і для кожного вказано, чим саме він стримується сьогодні.

In [ ]:
risks = pd.DataFrame([
    {
        "Ризик": "Впевнена неправильна формула",
        "Як проявляється": "Студент питає формулу, якої немає в базі. Модель «згадує» її "
                           "з пам'яті, відповідь виглядає так само надійно, як правильна.",
        "Чим контролюється": "Заборона в системному промпті + інструмент повертає явне "
                             "«немає в базі». Тест 5 і 6 перевіряють саме це.",
        "Залишковий ризик": "Промпт це не гарантія. Модель може порушити заборону, "
                            "і зловити це можна лише тестами.",
    },
    {
        "Ризик": "Правильна формула в неправильній задачі",
        "Як проявляється": "Формула дальності польоту застосована до кидання з даху: "
                           "відповідь занижена в півтора рази, помилки не видно.",
        "Чим контролюється": "Детермінований фільтр застосовності: предикати картки "
                             "звіряються з умовами задачі КОДОМ, до будь-якої генерації.",
        "Залишковий ризик": "Фільтр бачить лише ті ситуації, для яких є маркери. "
                            "Незнайоме формулювання пройде як «ознак порушення не знайдено».",
    },
    {
        "Ризик": "Прогалина в покритті бази",
        "Як проявляється": "Система відмовляє на половині запитів, продукт виглядає "
                           "непридатним, студент іде в Google.",
        "Чим контролюється": "Часткові збіги і підказки замість глухої відмови. "
                             "Логування запитів без відповіді як черга на поповнення бази.",
        "Залишковий ризик": "Головне обмеження продукту сьогодні. Вирішується не кодом, "
                            "а роботою над контентом.",
    },
    {
        "Ризик": "Неоднозначний запит",
        "Як проявляється": "«Формула дальності польоту» однаково описує два різні випадки, "
                           "і вибір за студента веде до неправильної відповіді.",
        "Чим контролюється": "Перевірка близькості оцінок: якщо різниця мала, система "
                             "показує варіанти і перепитує. Тест 4.",
        "Залишковий ризик": "Поріг близькості підібраний емпірично і може не спрацювати "
                            "на нових формулюваннях.",
    },
    {
        "Ризик": "Зростання вартості на довгих діалогах",
        "Як проявляється": "Історія йде в модель цілком щоразу, тому десятий хід "
                           "коштує помітно дорожче за перший.",
        "Чим контролюється": "Сьогодні ніяк, і це чесно зафіксовано як борг. "
                             "Наступний крок це обрізання історії після 6-8 ходів.",
        "Залишковий ризик": "На довгих сесіях рахунок росте непередбачувано.",
    },
    {
        "Ризик": "Інструмент підставив чужу формулу",
        "Як проявляється": "Запит «закон збереження імпульсу» резолвиться в найближчу "
                           "за словами картку (закон Ома) і отримує вердикт «придатна».",
        "Чим контролюється": "Гейт впевненості в ОБОХ інструментах: слабкий збіг веде "
                             "до відмови, а не до картки. Знайдено аудитом, закрито тестом.",
        "Залишковий ризик": "Поріг впевненості емпіричний. На новій формулі, схожій "
                            "за назвою на наявну, помилка може повторитися.",
    },
    {
        "Ризик": "Дрейф при зміні версії моделі",
        "Як проявляється": "Оновлення моделі змінює поведінку системи, у якій не змінено "
                           "жодного рядка коду.",
        "Чим контролюється": "Регресійний набір тестів: 12 сценаріїв, які ловлять зміну "
                             "поведінки. Плюс перевірка інструментів без мережі.",
        "Залишковий ризик": "Тести ловлять відоме. Нові класи помилок доведеться "
                            "знаходити так само, як я знайшов «закон Ома».",
    },
])

print("РИЗИКИ І КОНТРОЛЬ\n")
pd.set_option("display.max_colwidth", 62)
display(risks)

## Крок 10. Що я зрозумів, поки будував цю систему

Три спостереження з власної роботи, а не з матеріалів курсу. Кожне змінило
конкретне рішення в системі.

### 1. Небезпечна не помилка, а впевненість

Перша версія пошуку формул була односторонньою мірою схожості: скільки слів назви
знайшлося в запиті. Вона проходила всі мої тести, поки я не спитав про **закон
збереження енергії** і не отримав **ЗАКОН ОМА з оцінкою 1.0**, тобто як ідеальний збіг.

Причина виявилася дрібною: у назві «закон ома» слово «ома» коротке, фільтр коротких
слів його викидав, лишалося одне слово «закон», воно в запиті було, отже «збіглося все».

Мене вразила не сама помилка, а те, що система не мала **жодного способу
засумніватися**. Вона не вагалася, не показала альтернатив, не знизила впевненість.
Студент отримав би чужу формулу з тим самим тоном, що й правильну.

Звідси рішення, яке пройшло через усю систему: **міра схожості має падати, коли
даних мало, а не зростати від збігу одного загального слова**. Я замінив односторонню
міру на двосторонню (F2), де покриття запиту важить більше, і додав явний прапорець
впевненості. Тепер слабкий збіг веде до варіантів, а не до відповіді.

### 2. Ембеддінги розуміють тему, але не логіку

Я перевірив чисельно, як модель бачить два речення: «формула працює лише коли точка
кидання і точка падіння на одній висоті» і «камінь кидають з даху, тому початкова
висота не дорівнює нулю». Логічно це пряме протиріччя: друге описує ситуацію,
у якій перше забороняє застосовувати формулу. У векторному просторі вони близькі,
бо обидва про висоту і кидання.

Це закрило для мене питання, чи можна доручити перевірку застосовності
семантичному пошуку. **Не можна.** Тому в системі з'явився детермінований фільтр
з машиночитаними предикатами, який ухвалює рішення **до** будь-якої генерації.
Це найважливіший компонент продукту, і він принципово не використовує модель.

### 3. Тести теж уміють брехати

У моїй таблиці тестування два сценарії перевіряли, що система **не** викликає
інструментів (питання поза профілем). Одного разу я запустив набір з невалідним
ключем, і ці два тести **пройшли**: список викликаних інструментів порожній,
перевірка задоволена.

Тобто таблиця могла відрапортувати успіх при повністю непрацюючій системі.
Після цього я розділив «модель свідомо не викликала інструмент» і «виклик узагалі
не відбувся» на два різні вердикти. Урок ширший за один баг: **автотест, який не
відрізняє відсутність дії від відсутності системи, дає хибне відчуття контролю.**

## Крок 11. Шлях до production

Що вже готове, чого бракує і в якому порядку це закривати.

| Напрям | Стан сьогодні | Що потрібно для production |
|---|---|---|
| **Дані** | 12 формул, 6 з умовами застосовності | 300-500 карток на курс. Це головне обмеження, і воно вирішується не кодом, а роботою над контентом |
| **Retrieval** | гібридний: лексика плюс ембеддінги, RRF | re-ranker для топ-20, оцінка recall@k на розміченому наборі запитів |
| **Фільтр застосовності** | предикати плюс маркери ситуацій | розширити словник ситуацій, додати підтвердження розпізнаної умови у студента |
| **Контекст** | повна історія в кожному виклику | обрізання після 6-8 ходів, інакше вартість росте непередбачувано |
| **Тестування** | 12 сценаріїв плюс перевірка інструментів без мережі | розширити до 50+, додати регресію на кожен знайдений баг |
| **Спостережуваність** | лог викликаних інструментів | trace кожного запиту, метрики покриття бази, черга запитів без відповіді |
| **Інтерфейс** | Streamlit-прототип | автентифікація, збереження профілю студента, історія |

### Наступний крок, один

Якби треба було обрати **одну** дію, це не нова модель і не складніший агент,
а **вимірювання покриття бази на реальних запитах**. Сьогодні я не знаю головного
числа продукту: на якій частці запитів система відмовляє. Без нього неможливо
сказати, що робити далі, наповнювати базу чи міняти логіку пошуку.

Реалізація проста: логувати кожен запит, на який `formula_lookup` повернув
«немає в базі», і раз на тиждень дивитися топ. Це перетворює найбільше обмеження
продукту з відчуття на керовану чергу задач.

## Підсумок

StudyMate це не чатбот з формулами. Це система, побудована навколо одного продуктового
рішення: **впевнена неправильна відповідь небезпечніша за відмову**, бо студент
звернувся саме тому, що не може її перевірити.

З цього рішення випливає вся архітектура. Факти живуть у перевіреній базі, а не
в пам'яті моделі. Придатність формули вирішує код, а не семантична близькість.
Слабкий збіг веде до уточнення, а не до відповіді. Моделі лишається те, що вона
справді вміє, тобто мова і пояснення.

Ціна цього рішення теж чесна: система відмовляє частіше, ніж хотілося б, і головне
обмеження сьогодні це обсяг бази. Але відмова коштує студенту п'ять хвилин,
а впевнена помилка коштує оцінки на іспиті і довіри до продукту назавжди.

---

### Артефакти проєкту

- **Репозиторій:** https://github.com/Srh-Yakovenko-ua/AI_FUNDAMENTAL_FINAL
- **Веб-інтерфейс:** `app.py` (Streamlit), запуск описано в README
- **Попередні етапи:** ДЗ-2 продукт і архітектура, ДЗ-3 дані та retrieval,
  ДЗ-4 експеримент з ембеддінгами, ДЗ-5 агент на LangChain, ДЗ-6 агент на Agno

**Як запустити цей ноутбук:** відкрити в Google Colab, додати ключ OpenAI у панель
🔑 Secrets під іменем `OPENAI_API_KEY` (увімкнувши «Notebook access») і виконати
**Runtime → Run all**. Якщо Secrets недоступні, друга комірка запитає ключ через `getpass`.